In [ ]:
from pathlib import Path
import re
import datetime

# ===== Experiment setup =====
MODEL = "llama3.3-70b"
TEMPERATURE = 0
CONFIG_NAME = "C-Min"

scenario_name = "Car manufacturing supply chain management"


def slugify(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    return re.sub(r"_+", "_", s).strip("_")

# ===== Create log file =====
logs_dir = Path("logs")
logs_dir.mkdir(exist_ok=True)

stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
txt_path = logs_dir / f"{slugify(scenario_name)}__{slugify(CONFIG_NAME)}__{stamp}.txt"

with txt_path.open("w", encoding="utf-8") as f:
    f.write("EXPERIMENT LOG\n")
    f.write("=" * 70 + "\n")
    f.write(f"SCENARIO: {scenario_name}\n")

    f.write(f"CONFIG: {CONFIG_NAME}\n")
    f.write(f"MODEL: {MODEL}\n")
    f.write(f"TEMPERATURE: {TEMPERATURE}\n")
    f.write(f"CREATED_AT: {stamp}\n")
    f.write("=" * 70 + "\n\n")

print("Log file created:", txt_path)


Log file created: logs/medical_appointment_platform__c_min__20260203-111720.txt


In [7]:
from openai import OpenAI

client = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="local-dummy")




partial_model = """
Elements:
    Classes:
    - User

"""

prompt = f"""
 You are a modeling assistant. Complete a partial UML class diagram for the scenario.

Scenario: {scenario_name}
Current partial model:
{partial_model}

Task:
Propose up to 3 additions to extend the model. an addition could be of a  :
- type: ARCH (new class) or CONN (new association), or CHAR (new attribute/constraint/property)


Write as a bullet list. Be concise.
""".strip()

print("===== PROMPT =====")
print(prompt)

resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": prompt}],
    temperature=TEMPERATURE,
)

output_text = resp.choices[0].message.content

print("\n===== MODEL OUTPUT =====")
print(output_text)


===== PROMPT =====
You are a modeling assistant. Complete a partial UML class diagram for the scenario.

Scenario: Medical appointment platform
Current partial model:

Elements:
    Classes:
    - User



Task:
Propose up to 3 additions to extend the model. an addition could be of a  :
- type: ARCH (new class) or CONN (new association), or CHAR (new attribute/constraint/property)


Write as a bullet list. Be concise.

===== MODEL OUTPUT =====
* ARCH: Appointment (new class)
* CONN: User - Appointment (new association, e.g., "has" or "schedules")
* CHAR: username (new attribute of User class, for unique identification)


In [8]:
# ===== Manually fill this dict (we copy/paste items from output) =====
classified = {
    "arch": [
       "Appointment", 
    ],
    "connections": [
        "user - appointment",
    ],
    "characteristics": [
       "username"
    ],
}

iteration_id = 1
# ===== Log everything (prompt + output + youand the  manual dict we accepted to integrate) =====
with txt_path.open("a", encoding="utf-8") as f:
    f.write(f"\n\nITERATION {iteration_id}\n")
    f.write("-" * 80 + "\n")

    f.write("PARTIAL MODEL\n")
    f.write("-" * 80 + "\n")
    f.write(partial_model.strip() + "\n\n")

    f.write("PROMPT\n")
    f.write("-" * 80 + "\n")
    f.write(prompt + "\n\n")

    f.write("MODEL OUTPUT\n")
    f.write("-" * 80 + "\n")
    f.write(output_text.strip() + "\n\n")

    f.write("MANUAL CLASSIFICATION\n")
    f.write("-" * 80 + "\n")
    f.write("ARCH:\n")
    for x in classified["arch"]:
        f.write(f"- {x}\n")
    f.write("\nCONNECTIONS:\n")
    for x in classified["connections"]:
        f.write(f"- {x}\n")
    f.write("\nCHARACTERISTICS:\n")
    for x in classified["characteristics"]:
        f.write(f"- {x}\n")

print(f"\nLogged iteration {iteration_id} to:", txt_path)
classified


Logged iteration 1 to: logs/medical_appointment_platform__c_min__20260203-111720.txt


{'arch': ['Appointment'],
 'connections': ['user - appointment'],
 'characteristics': ['username']}

In [9]:
# ===== Manually fill accepted items for this iteration =====
accepted = {
    "arch": [
        "Appointment",
    ],
    "connections": [
    
    ],
    "characteristics": [
       "username"
    ],
}

# ===== Append accepted to the same log file =====
with txt_path.open("a", encoding="utf-8") as f:
    f.write("\n\nACCEPTED (ITERATION {})\n".format(iteration_id))
    f.write("-" * 80 + "\n")

    f.write("ARCH:\n")
    for x in accepted["arch"]:
        f.write(f"- {x}\n")

    f.write("\nCONNECTIONS:\n")
    for x in accepted["connections"]:
        f.write(f"- {x}\n")

    f.write("\nCHARACTERISTICS:\n")
    for x in accepted["characteristics"]:
        f.write(f"- {x}\n")

print(f"Accepted suggestions appended for iteration {iteration_id} to:", txt_path)
accepted


Accepted suggestions appended for iteration 1 to: logs/medical_appointment_platform__c_min__20260203-111720.txt


{'arch': ['Appointment'], 'connections': [], 'characteristics': ['username']}